In [9]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [12]:
import os
print(os.listdir('/content/drive/MyDrive/'))

['Colab Notebooks', 'tweets (5).gsheet', 'tweets_clean.csv', 'airline_model', 'predictions.csv', 'bertopic_output.csv']


In [13]:
print(os.listdir('/content/drive/MyDrive/airline_model'))

['model.safetensors', 'tokenizer_config.json', 'tokenizer.json', 'config.json']


In [14]:
model_path = '/content/drive/MyDrive/airline_model'

tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
model = AutoModelForSequenceClassification.from_pretrained(model_path, local_files_only=True, ignore_mismatched_sizes=True)
print('Model loaded!')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded!


In [15]:
df = pd.read_csv('/content/drive/MyDrive/tweets_clean.csv')
print(df.shape)

(12679, 10)


In [17]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
all_predictions = []
for i in range(0, len(df), 32):
    batch = df['text_clean'][i:i+32].tolist()
    inputs = tokenizer(batch, padding=True, truncation=True,
                       max_length=128, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        preds = outputs.logits.argmax(dim=1).cpu().tolist()
    all_predictions.extend(preds)

df['predicted_label'] = all_predictions
print(df['predicted_label'].value_counts())

predicted_label
0    8341
1    2520
2    1818
Name: count, dtype: int64


In [18]:
df['tweet_length']= df['text'].str.len()
df['word_count'] = df['text_clean'].str.split().str.len()
df['exclamation_count'] = df['text'].str.count('!')
df['has_url'] = df['text'].str.contains('http').astype(int)
df['mention_count'] = df['text'].str.count(r'@\w+')
df['tweet_hour'] = pd.to_datetime(df['tweet_created']).dt.hour
df[['tweet_length','word_count','exclamation_count',
    'has_url','mention_count','tweet_hour']].head(20)

,tweet_length,word_count,exclamation_count,has_url,mention_count,tweet_hour
0,71,8,1,0,1,11
1,126,10,0,0,1,11
2,55,4,0,0,1,11
3,135,11,0,0,1,11
4,85,9,0,0,1,11
5,80,6,0,0,1,10
6,95,8,0,0,1,10
7,83,6,0,0,1,10
8,139,11,2,0,2,10
9,45,4,0,0,1,10


In [19]:
import numpy as np

df['score'] = 0
df['score'] += (df['tweet_length'] > 100).astype(int)
df['score'] += (df['exclamation_count'] >= 2).astype(int)
df['score'] += (df['has_url'] == 1).astype(int)
df['score'] += (df['mention_count'] > 1).astype(int)
df['score'] += (df['negativereason'].isin(['Lost Luggage', 'Cancelled Flight', 'Customer Service Issue'])).astype(int)

df['churn'] = np.where(
    (df['predicted_label'] == 0) & (df['score'] >= 3), 1, 0
)

print(df['churn'].value_counts())

churn
0    12004
1      675
Name: count, dtype: int64


In [21]:
from sklearn.model_selection import train_test_split
X = df[['tweet_length', 'word_count', 'exclamation_count',
    'has_url', 'mention_count', 'tweet_hour', 'predicted_label']]
y = df['churn']
X_train, X_test, y_train, y_test = train_test_split( X,y,test_size=0.2, random_state=42, stratify=y)
print(X_train.shape)
print(X_test.shape)

(10143, 7)
(2536, 7)


In [24]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

In [25]:
model = XGBClassifier(scale_pos_weight=17.78)
model.fit(X_train, y_train)
predictions = model.predict(X_test)
print(classification_report(y_test, predictions,
      target_names=['low risk', 'high churn risk']))

                 precision    recall  f1-score   support

       low risk       0.99      0.96      0.97      2401
high churn risk       0.53      0.83      0.65       135

       accuracy                           0.95      2536
      macro avg       0.76      0.89      0.81      2536
   weighted avg       0.97      0.95      0.96      2536



In [26]:
from sklearn.model_selection import GridSearchCV

In [27]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 6, 9],
    'learning_rate': [0.1, 0.2, 0.3]
}

In [29]:
# Create GridSearchCV
grid_search = GridSearchCV(XGBClassifier(scale_pos_weight=17.78), param_grid, cv=3, scoring='f1')

# Fit
grid_search.fit(X_train, y_train)

# Print results
print(grid_search.best_params_)
print(grid_search.best_score_)

{'learning_rate': 0.3, 'max_depth': 3, 'n_estimators': 100}
0.689009874562247


In [30]:
final_model = XGBClassifier(
    scale_pos_weight=17.78,
    learning_rate=0.3,
    max_depth=3,
    n_estimators=100,
    random_state=42
)
final_model.fit(X_train, y_train)
final_preds = final_model.predict(X_test)
print(classification_report(y_test, final_preds,
      target_names=['low risk', 'high churn risk']))

                 precision    recall  f1-score   support

       low risk       1.00      0.95      0.97      2401
high churn risk       0.53      0.96      0.68       135

       accuracy                           0.95      2536
      macro avg       0.76      0.96      0.83      2536
   weighted avg       0.97      0.95      0.96      2536



In [32]:
import joblib
joblib.dump(final_model, '/content/drive/MyDrive/churn_model.pkl')
print('Tuned churn model saved!')

Tuned churn model saved!
